# Stage 05a: Initial Model Training

**Purpose:** Quick validation with default hyperparameters

**Features:** Monotonicity constraints, exclusions, GLM base_margin

**Outputs:** models/05a_model_initial.json, results/05a_*

In [ ]:
config_path = "config/car_coll/v1"

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for papermill

import pandas as pd
import numpy as np
# STAGE 05a: INITIAL MODEL TRAINING
import yaml
import os
import sys
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error

sys.path.insert(0, str(Path.cwd() / 'lib'))
from model_utils import load_monotonicity_constraints, load_exclusions, load_glm_init, apply_feature_filters, build_xgb_params
from utils import setup_notebook_environment

print("########################################")
print("# STAGE 05: MODEL TRAINING")
print("########################################")

project_root = setup_notebook_environment()

In [ ]:
print(f"Python: {sys.version}")
print(f"XGBoost: {xgb.__version__}")


In [ ]:
config_file = f"{config_path}/config.yaml"
with open(config_file, 'r') as f:
    cfg = yaml.safe_load(f)

output_base = cfg['paths']['output_base']
target = cfg['experiment']['target']
exposure = cfg['experiment']['exposure']
vehicle_type = cfg['experiment']['vehicle_type']
print(f"Vehicle type: {vehicle_type}")

In [ ]:
# Load ENCODED train/test data from Stage 04c
train_encoded = pd.read_parquet(f"{output_base}/data/04c_train_encoded.parquet")
test_encoded = pd.read_parquet(f"{output_base}/data/04c_test_encoded.parquet")

# Load original data for target/exposure
train_orig = pd.read_parquet(f"{output_base}/data/04_train.parquet")
test_orig = pd.read_parquet(f"{output_base}/data/04_test.parquet")

print(f"\n* Train encoded: {train_encoded.shape}")
print(f"* Test encoded: {test_encoded.shape}")

In [ ]:
# Use ALL encoded features (encoding already applied in 04c)
X_train = train_encoded
y_train = train_orig[target]
X_test = test_encoded
y_test = test_orig[target]

print(f"\n* Features: {X_train.shape[1]} encoded features")

In [ ]:
# Train XGBoost model
print(f"\n* Training XGBoost...")

xgb_params = cfg['xgboost'].copy()
n_estimators = xgb_params.pop('n_estimators')

# Construct full eval_metric for tweedie if needed
if 'eval_metric' in xgb_params and 'tweedie' in xgb_params['eval_metric']:
    if '@' not in xgb_params['eval_metric']:
        variance_power = xgb_params['tweedie_variance_power']
        xgb_params['eval_metric'] = f"{xgb_params['eval_metric']}@{variance_power}"

model = xgb.XGBRegressor(n_estimators=n_estimators, **xgb_params)
model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100
)

print(f"\n* Model trained")
print(f"* Best iteration: {model.best_iteration}")

In [ ]:
# Save training history
results = model.evals_result()
metric_base = cfg['xgboost']['eval_metric']

# Construct full metric name for tweedie if needed
if 'tweedie' in metric_base and '@' not in metric_base:
    variance_power = cfg['xgboost']['tweedie_variance_power']
    metric = f"{metric_base}@{variance_power}"
else:
    metric = metric_base

# Use exact metric from results
result_key = list(results['validation_0'].keys())[0]

history_df = pd.DataFrame({
    'iteration': range(len(results['validation_0'][result_key])),
    f'train_{metric}': results['validation_0'][result_key],
    f'test_{metric}': results['validation_1'][result_key]
})

history_file = f"{output_base}/results/05a_training_history.parquet"
history_df.to_parquet(history_file, index=False)
print(f"\n* Training history saved: {history_file}")
print(f"  Total iterations: {len(history_df)}")
print(f"  Metric: {metric}")
print(f"  Final train {metric}: {history_df[f'train_{metric}'].iloc[-1]:.4f}")
print(f"  Final test {metric}: {history_df[f'test_{metric}'].iloc[-1]:.4f}")

In [ ]:
# Plot learning curve
from utils import plot_learning_curve
from IPython.display import Image, display

plot_path = plot_learning_curve(history_df, f"{output_base}/plots/05a_learning_curve.png", metric=metric)
print(f"\n* Learning curve saved: {plot_path}")

# Display inline
display(Image(filename=plot_path))

In [ ]:
# Predictions
train_orig['pred'] = model.predict(X_train)
test_orig['pred'] = model.predict(X_test)

# Metrics
train_mae = mean_absolute_error(y_train, train_orig['pred'])
test_mae = mean_absolute_error(y_test, test_orig['pred'])
train_rmse = np.sqrt(mean_squared_error(y_train, train_orig['pred']))
test_rmse = np.sqrt(mean_squared_error(y_test, test_orig['pred']))

print(f"\n* Metrics:")
print(f"  Train MAE: {train_mae:.4f}")
print(f"  Test MAE: {test_mae:.4f}")
print(f"  Train RMSE: {train_rmse:.4f}")
print(f"  Test RMSE: {test_rmse:.4f}")

In [ ]:
# Save model
model_file = f"{output_base}/models/xgb_model.json"
os.makedirs(f"{output_base}/models", exist_ok=True)
model.get_booster().save_model(model_file)
print(f"\n* Model saved: {model_file}")

In [ ]:
# Save metrics
metrics = {
    'train_mae': float(train_mae),
    'test_mae': float(test_mae),
    'train_rmse': float(train_rmse),
    'test_rmse': float(test_rmse),
    'n_features': X_train.shape[1],
    'train_size': len(train_orig),
    'test_size': len(test_orig)
}

metrics_file = f"{output_base}/results/05a_metrics.yaml"
with open(metrics_file, 'w') as f:
    yaml.dump(metrics, f)

print(f"* Metrics saved: {metrics_file}")

In [ ]:
# Save predictions separately for train and test
# Train predictions
train_pred = train_orig[[target, exposure, 'pred']].copy()
train_pred.columns = ['actual', 'exposure', 'pred']

# Validate predictions output
required_cols = ['actual', 'exposure', 'pred']
missing = [c for c in required_cols if c not in train_pred.columns]
if missing:
    raise ValueError(f"[Stage 05] Missing prediction columns: {missing}")

train_pred_file = f"{output_base}/results/05a_predictions_train.parquet"
train_pred.to_parquet(train_pred_file)

# Test predictions
test_pred = test_orig[[target, exposure, 'pred']].copy()
test_pred.columns = ['actual', 'exposure', 'pred']
test_pred_file = f"{output_base}/results/05a_predictions_test.parquet"
test_pred.to_parquet(test_pred_file)

print(f"* Train predictions saved: {train_pred_file}")
print(f"* Test predictions saved: {test_pred_file}")
print(f"[Stage 05 Output] ✓ Prediction columns validated: {', '.join(required_cols)}")

## Lift Charts

In [ ]:
# Prepare data for lift charts
# Get exposure column from config, or use fallback logic
exposure_col = cfg['experiment'].get('exposure', None)

if exposure_col and exposure_col in train_orig.columns:
    weight_col = exposure_col
    print(f"\n* Using '{exposure_col}' from config as weight column")
elif 'ee' in train_orig.columns:
    weight_col = 'ee'
    print(f"\n* Using 'ee' (earned exposure) as weight column")
elif 'ee_imps' in train_orig.columns:
    weight_col = 'ee_imps'
    print(f"\n* Using 'ee_imps' (earned exposure imputed) as weight column")
else:
    train_orig['weight'] = 1  # Equal weight per record
    test_orig['weight'] = 1
    weight_col = 'weight'
    print(f"\n* WARNING: No exposure column found, using equal weighting (weight=1)")
    if exposure_col:
        print(f"  Config specifies '{exposure_col}' but it's not in the data!")

# Create incurred/denom columns for lift chart
# Use exposure as denominator if available
if weight_col in train_orig.columns and weight_col != 'weight':
    train_orig['incurred_act'] = train_orig[target]
    train_orig['incurred_pred'] = train_orig['pred']
    train_orig['denom'] = train_orig[weight_col]  # Use exposure as denominator
    
    test_orig['incurred_act'] = test_orig[target]
    test_orig['incurred_pred'] = test_orig['pred']
    test_orig['denom'] = test_orig[weight_col]
else:
    train_orig['incurred_act'] = train_orig[target]
    train_orig['incurred_pred'] = train_orig['pred']
    train_orig['denom'] = 1  # Equal weight
    
    test_orig['incurred_act'] = test_orig[target]
    test_orig['incurred_pred'] = test_orig['pred']
    test_orig['denom'] = 1

print(f"* Prepared data for lift charts (weight_col='{weight_col}')")

In [ ]:
# Load optimized lift chart function
import matplotlib.pyplot as plt
from lift_chart_fast import create_lift_chart

print("* Optimized lift chart function loaded")

In [ ]:
print("\n* Generating train lift chart...")


In [ ]:
fig_train_orig, table_train = create_lift_chart(train_orig, weight_col, bins=10, title="Train Lift Chart")

In [ ]:
# Train lift chart


train_chart_file = f"{output_base}/results/05a_lift_chart_train.png"
fig_train_orig.savefig(train_chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_train_orig)

print(f"  Saved: {train_chart_file}")

# Display inline in notebook
from IPython.display import Image, display
display(Image(train_chart_file))

print("\nTrain decile table:")
print(table_train[['decile', 'act', 'pred', 'act_rel', 'pred_rel']])

In [ ]:
# Test lift chart
print("\n* Generating test lift chart...")
fig_test_orig, table_test = create_lift_chart(test_orig, weight_col, bins=10, title="Test Lift Chart")

test_chart_file = f"{output_base}/results/05a_lift_chart_test.png"
fig_test_orig.savefig(test_chart_file, dpi=150, bbox_inches='tight')
plt.close(fig_test_orig)

print(f"  Saved: {test_chart_file}")

# Display inline in notebook
display(Image(test_chart_file))

print("\nTest decile table:")
print(table_test[['decile', 'act', 'pred', 'act_rel', 'pred_rel']])

In [ ]:
print("\n########################################")
print("# STAGE 05: COMPLETE")
print("########################################")